In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 5.4 Microstates, Entropy, Temperature, and the Boltzmann Distribution

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume V — Classical Statistical Mechanics",
    number="5.4",
    title="Microstates, Entropy, Temperature, and the Boltzmann Distribution",
    blurb="Where counting becomes physics. From the postulate that every "
    "accessible microstate is equally likely — the microcanonical ensemble — "
    "we count our way to entropy, watch the second law emerge as a system "
    "finding its most probable state, discover what temperature actually is, "
    "and derive the Boltzmann factor that governs everything that follows.",
    difficulty="advanced",
    estimate="160–200 min",
)

## Notebook overview

This is the first *physics* notebook of Volume V, and it makes good on a promise the math
arsenal has been building toward: that **counting microstates is doing statistical
mechanics**. From a single postulate — every accessible microstate of an isolated system is
equally likely — and the combinatorics of [§5.1](counting.ipynb)–[§5.3](large-n-limit.ipynb), we will construct the entire conceptual
core of the subject. The ensemble of equally-probable microstates of an isolated system, at
fixed energy, volume, and particle number, has a name we will use throughout: the
**microcanonical ensemble**. Entropy will turn out to be the logarithm of its multiplicity.
The second law will turn out to be nothing more than a system drifting to its most probable
macrostate. Temperature will *emerge*, not be assumed, as the energy-derivative of entropy.
And the Boltzmann factor $e^{-E/kT}$, the formula the rest of statistical mechanics runs on,
will fall out of counting a reservoir's microstates.

We make this concrete with the **two-state paramagnet**: $N$ classical spins in a magnetic
field, each pointing either up or down. It is a genuinely classical system — "a spin that
points two ways" is an ordinary modelling assumption, needing no quantum mechanics to posit —
and its multiplicity is the binomial coefficient we counted in [§5.1](counting.ipynb), so every step of the
argument is exact. The arsenal is everywhere here, and we flag it as we go: the multiplicity is
the $\binom{N}{N_\uparrow}$ of [§5.1](counting.ipynb); we compute it in log space with `scipy.special.gammaln` because
the counts overflow (the lesson of [§5.3](large-n-limit.ipynb) and [§0.1](../00-foundations/floating-point.ipynb)); the equilibrium peak is razor-sharp for the
$1/\sqrt N$ reason of [§5.3](large-n-limit.ipynb); and entropy is the log-multiplicity we first met in [§5.3](large-n-limit.ipynb). What was
abstract there becomes physical here.

We will (1) meet microstates and the postulate on a tiny spin system, (2) derive the paramagnet
multiplicity, (3) read entropy off it, (4) bring two paramagnets into thermal contact and find
the sharp equilibrium peak, (5) watch the second law emerge as energy relaxes to that peak, (6)
extract temperature as $1/T=\partial S/\partial E$, (7) derive the Boltzmann distribution from a
reservoir, and (8) push the definition to its breaking point with negative temperature.

> **How to read the checks.** Each exercise closes with a `validate` call against an
> independent fact: macrostate probabilities equal to multiplicities; the paramagnet
> $\binom{N}{N_\uparrow}$; extensive entropy maximal at half-filling; the equilibrium partition
> at proportional energy sharing; the total entropy maximized there; equal $\partial S/\partial
> E$ defining temperature; the Boltzmann $e^{-E/kT}$ from reservoir counting; a negative slope
> past half-filling. A ✓ is strong evidence; a ✗ is a prompt to *locate the discrepancy*, not a
> verdict.
>
> **Scope.** The microcanonical ensemble and the conceptual core of statistical mechanics from
> counting; why an ensemble average over phase space is valid at all is [§5.5](ergodicity.ipynb) (ergodicity), and the
> canonical ensemble that systematizes the Boltzmann bridge comes later in the volume. See
> Schroeder, *An Introduction to Thermal Physics*; Kardar, *Statistical
> Physics of Particles*; and [§5.1](counting.ipynb) (the binomial count), [§5.3](large-n-limit.ipynb) (log space, entropy,
> $1/\sqrt N$).

## Theory in brief

### Microstates, macrostates, and the microcanonical ensemble

A **microstate** is a complete specification of a system; a **macrostate** is a coarse
description (a total energy, a particle number) compatible with many microstates. The number
of microstates realizing a macrostate is its **multiplicity** $\Omega$ — the counting of [§5.1](counting.ipynb),
now physical. The whole subject rests on one assumption,

```{math}
:label: eq-postulate
\text{an isolated system in equilibrium is equally likely to be in any accessible microstate,}
```

and the collection of those equally-probable microstates of an isolated system (fixed $E$, $V$,
$N$) is the **microcanonical ensemble**. A macrostate's probability is then proportional to its
multiplicity {eq}`eq-micro-macro`,

```{math}
:label: eq-micro-macro
P(\text{macrostate})=\frac{\Omega(\text{macrostate})}{\Omega_{\rm total}} .
```

### The two-state paramagnet

Our exactly-countable model is $N$ classical spins in a magnetic field, each up ($\sigma=+1$,
energy $+\varepsilon$) or down ($\sigma=-1$, energy $0$). The energy is set by the number
pointing up, $E=N_\uparrow\,\varepsilon$, and the number of microstates with $N_\uparrow$ up is
the count of *which* spins are up — a binomial coefficient,

```{math}
:label: eq-paramagnet
\Omega(N,N_\uparrow)=\binom{N}{N_\uparrow} ,
```

exactly the combinations of [§5.1](counting.ipynb). The ground state is all spins down ($N_\uparrow=0$, one
microstate); the counts overflow almost at once, so we work with $\ln\Omega$ via
`scipy.special.gammaln` (the [§5.3](large-n-limit.ipynb) lesson).

### Entropy

Boltzmann's formula, the log-multiplicity of [§5.3](large-n-limit.ipynb) made physical,

```{math}
:label: eq-microstates-entropy
S=k\ln\Omega ,
```

is extensive (scales with system size). For the paramagnet it is *maximal at half-filling*
($N_\uparrow=N/2$, the most ways to arrange the spins) and vanishes at the extremes.

### Thermal contact and the second law

Bring two paramagnets $A$ and $B$ into contact, free to exchange energy at fixed total
$N_\uparrow=N_\uparrow^A+N_\uparrow^B$. The total multiplicity is a product,

```{math}
:label: eq-second-law
\Omega_{\rm total}(N_\uparrow^A)=\Omega_A(N_\uparrow^A)\,\Omega_B(N_\uparrow-N_\uparrow^A) ,
```

and it is *sharply* peaked (the $1/\sqrt N$ sharpness of [§5.3](large-n-limit.ipynb)) at the partition that maximizes
$S_A+S_B$. Energy flows to maximize the total multiplicity: the **second law** is not a
separate postulate but the statement that an isolated system drifts to its most probable
macrostate.

### Temperature

At the peak, $\partial S_A/\partial E_A=\partial S_B/\partial E_B$. The quantity that equalizes
when energy stops flowing *defines* temperature,

```{math}
:label: eq-temperature
\frac{1}{T}=\frac{\partial S}{\partial E} .
```

A body with the smaller $\partial S/\partial E$ (the hotter one) gives up energy.

### The Boltzmann distribution

Put a small system in contact with a large reservoir at temperature $T$. By the postulate, the
probability the small system sits in a microstate of energy $E$ is proportional to the number
of reservoir microstates left, $\Omega_{\rm res}(E_{\rm tot}-E)$. Since $\ln\Omega_{\rm res}$
falls linearly in the energy handed over, with slope $1/kT$,

```{math}
:label: eq-boltzmann
P(E)\propto\Omega_{\rm res}(E_{\rm tot}-E)\propto e^{-E/kT} .
```

This is the master formula of statistical mechanics, and it is a counting result.

## Setup

In [ ]:
from math import comb

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from scipy.special import gammaln

from ecp import combinatorics as cb
from ecp import draw, validate
from ecp.animate import show

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = cb.RED


def ln_omega_paramagnet(N_up, N):
    """The log-multiplicity ln Ω(N, N_up) = ln C(N, N_up) of a two-state paramagnet (eq-paramagnet).

    The number of ways to choose which N_up of N classical spins point up — *exactly*
    the binomial / combinations count of §5.1. Computed in log space with
    ``scipy.special.gammaln`` (which gives ln Γ) because the multiplicity itself
    overflows for any macroscopic system (the §5.3 lesson).

    Parameters
    ----------
    N_up : int or numpy.ndarray
        Number of up-spins (the energy is N_up·ε).
    N : int
        Total number of spins.

    Returns
    -------
    float or numpy.ndarray
        The log-multiplicity ln Ω.
    """
    N_up = np.asarray(N_up, dtype=float)
    return gammaln(N + 1.0) - gammaln(N_up + 1.0) - gammaln(N - N_up + 1.0)


def spin_row(ax, up_mask, x0=0.0, y=0.0, pitch=0.5):
    """Draw a row of up/down spin arrows, a schematic of a paramagnet configuration.

    Each site is an arrow pointing up (aligned, energy +ε, amber) or down
    (energy 0, ink). The arrows are inline glyphs, exempt from the collision gate.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axes to draw on.
    up_mask : sequence of bool
        Per-site spin direction (``True`` = up).
    x0, y : float, optional
        Lower-left position of the first spin.
    pitch : float, optional
        Horizontal spacing between spins (default ``0.5``).

    Returns
    -------
    None
        Draws the spin row onto ``ax`` in place.
    """
    for i, up in enumerate(up_mask):
        ax.text(
            x0 + i * pitch,
            y,
            r"$\uparrow$" if up else r"$\downarrow$",
            fontsize=20,
            ha="center",
            va="center",
            color=ACCENT if up else INK,
            gid="_nocheck",
        )

## Exercise 1 — Microstates, macrostates, and the postulate

We start with the one idea everything rests on, on a system small enough to write out in full.
Take three spins, each up or down. A **microstate** is a complete specification — which spin
points which way — and there are $2^3=8$ of them. A **macrostate** is the coarse description we
actually care about, here the *number* of up-spins (which fixes the energy), and there are only
four: $0,1,2,3$. Each macrostate is compatible with several microstates, and the count is the
multiplicity: $1,3,3,1$. The **fundamental postulate** {eq}`eq-postulate` says every microstate
is equally likely — this is the **microcanonical ensemble** — so each of the eight has
probability $1/8$, and therefore a macrostate's probability is simply its multiplicity over the
total {eq}`eq-micro-macro` ({numref}`fig-me-postulate`). The most probable macrostate is the one
realised the most ways. That single sentence, scaled to $10^{23}$, is the whole of equilibrium
statistical mechanics.

**This exercise (worked).** Enumerate the eight microstates of three spins (with
`itertools.product`), group them by number of up-spins, confirm the enumerated multiplicities
are the binomial coefficients $C(3,k)=1,3,3,1$ of [§5.1](counting.ipynb) (with `math.comb`), and confirm that
the macrostate probabilities are the multiplicities divided by the total — the postulate in
action.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    multiplicity.tolist() == [comb(3, k) for k in range(4)]
    and len(microstates) == 2**3
    and np.allclose(P_macro, multiplicity / 8.0),
    "the enumerated multiplicities are the binomial C(3,k)=1,3,3,1 and set the macrostate probabilities",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — The paramagnet multiplicity

To do physics we need a model we can count exactly, and the **two-state paramagnet** is the
cleanest one there is: $N$ classical spins in a magnetic field, each pointing up or down
({numref}`fig-me-paramagnet`). With each up-spin carrying energy $\varepsilon$ and each down-spin
zero, a macrostate is fixed by the number of up-spins $N_\uparrow$ (which sets the energy
$E=N_\uparrow\varepsilon$), and its multiplicity is the number of ways to choose *which*
$N_\uparrow$ of the $N$ spins are up. That is exactly the combinations count of [§5.1](counting.ipynb), the
binomial coefficient $\Omega=\binom{N}{N_\uparrow}$ {eq}`eq-paramagnet`. Nothing here is quantum:
a spin that points two ways is an ordinary classical modelling assumption. The numbers explode
immediately — a modest magnet has a multiplicity with thousands of digits — so, exactly as [§5.3](large-n-limit.ipynb)
warned, we never compute $\Omega$ itself but its logarithm, with `scipy.special.gammaln`.

**This exercise (worked).** Confirm the paramagnet multiplicity equals the binomial count
$\binom{N}{N_\uparrow}$ for small $N$ with `math.comb`, verify the `ln_omega_paramagnet` helper
reproduces $\ln\Omega$, and show that for a realistic magnet ($N=4000$ spins, half up) the count
overflows a float while its logarithm stays finite.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    ln_omega_paramagnet(N_up_small, N_small),
    np.log(float(comb(N_small, N_up_small))),
    "the paramagnet multiplicity is the binomial count C(N, N↑)",
    rtol=1e-9,
)
validate.check(
    overflowed and np.isfinite(ln_omega_paramagnet(N_big // 2, N_big)),
    "the macroscopic multiplicity overflows a float; its logarithm via gammaln does not",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — Entropy as log-multiplicity

With the multiplicity in hand, entropy costs us nothing new: it is **Boltzmann's formula**
$S=k\ln\Omega$ {eq}`eq-microstates-entropy`, the log-multiplicity of [§5.3](large-n-limit.ipynb) with a name and a constant. The
paramagnet's entropy has a telling shape ({numref}`fig-me-entropy`). It *vanishes at the
extremes* — all spins down or all up is a single microstate, $\Omega=1$, $S=0$ — and is
*maximal at half-filling* $N_\uparrow=N/2$, where there are the most ways to arrange the spins.
Between, it rises and falls, the entropy of mixing of [§5.3](large-n-limit.ipynb) in physical dress. And it is
**extensive**: a magnet twice as large at the same fraction of up-spins has twice the entropy,
because $\ln\Omega$ scales with system size (a fact Stirling makes exact in the large-$N$ limit
of [§5.3](large-n-limit.ipynb)). Extensivity is what lets entropy, like energy, be summed over the parts of a composite
system — the property we lean on in the next exercise.

**This exercise (worked).** Compute $S/k=\ln\Omega(N,N_\uparrow)$ across $N_\uparrow$ (the
`ln_omega_paramagnet` helper); confirm it is maximal at $N_\uparrow=N/2$ with $S_{\max}=\ln
\binom{N}{N/2}$ and vanishes at the extremes; then confirm extensivity by checking $\ln\Omega(2N,
2N_\uparrow)\approx2\ln\Omega(N,N_\uparrow)$ at large $N$.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    np.argmax(S_over_k) == N_e // 2 and S_over_k[0] < 1e-9 and S_over_k[-1] < 1e-9,
    "the entropy is maximal at half-filling (N↑=N/2) and vanishes at the extremes",
)
validate.close(
    S_double / S_single,
    2.0,
    "entropy is extensive — it scales with system size",
    rtol=1e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — Two paramagnets in thermal contact: the sharp peak

Now the centerpiece. Place two paramagnets $A$ and $B$ side by side, able to exchange energy —
an up-spin in one can flip down while a down-spin in the other flips up, moving one unit of
energy across ({numref}`fig-me-contact`). The total energy $N_\uparrow=N_\uparrow^A+N_\uparrow^B$
is fixed, but it can be partitioned any way. By the postulate every microstate of the combined
system is equally likely, so the probability of a given partition $N_\uparrow^A$ is proportional
to the total multiplicity $\Omega_{\rm total}(N_\uparrow^A)=\Omega_A(N_\uparrow^A)\,\Omega_B
(N_\uparrow-N_\uparrow^A)$ {eq}`eq-second-law` — a product of two large, opposing binomials. The
result is a function *spectacularly* peaked at one partition ({numref}`fig-me-peak`), for
precisely the $1/\sqrt N$ reason of [§5.3](large-n-limit.ipynb): the product of two multiplicities is as sharp as the
binomial was. And the peak sits exactly where energy is shared *in proportion to size*,
$N_\uparrow^{A\star}=N_\uparrow\,N_A/(N_A+N_B)$.

**This exercise (worked).** For $N_A=N_B=100$ spins sharing $N_\uparrow=60$ units of energy,
compute $\ln\Omega_{\rm total}(N_\uparrow^A)$ in log space, find the peak with `numpy.argmax`, and
confirm it is at the proportional partition $N_\uparrow^{A\star}=30$; measure the relative width
and confirm it shrinks as $1/\sqrt N$. Repeat for unequal sizes $N_A=50,N_B=150$ at $N_\uparrow=
80$ and confirm the peak moves to $N_\uparrow^{A\star}\approx20$ — the larger magnet takes the
larger share.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    up_peak,
    up_tot * N_A / (N_A + N_B),
    "energy is shared in proportion to system size at the multiplicity peak",
    atol=2,
)
validate.close(
    up_peak2,
    up_tot2 * N_A2 / (N_A2 + N_B2),
    "the asymmetric paramagnets also share energy in proportion to size",
    atol=2,
)

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


## Exercise 5 — The second law from counting

Here is where a deep idea becomes simple. The **second law of thermodynamics** says the entropy
of an isolated system never decreases. In our picture that is almost a tautology: since
$S_{\rm total}=S_A+S_B=k\ln\Omega_{\rm total}$ {eq}`eq-microstates-entropy`, maximizing the total entropy is
the *same* as maximizing the total multiplicity, which is the same as finding the most probable
partition. So when we start the two magnets with an unequal split and let energy flow, it flows
in the direction that increases $\Omega_{\rm total}$ — overwhelmingly likely, because that
direction has vastly more microstates — until it reaches the peak and stops
({numref}`fig-me-relax`). The second law is not an extra postulate; it is the statement that an
isolated system drifts to its most probable macrostate, and "irreversibility" is just the
fantastically long odds against drifting back.

**This exercise (worked).** Confirm that $S_{\rm total}(N_\uparrow^A)$ is maximized at the same
partition the multiplicity peaks at (`numpy.argmax`); then *simulate* the approach to
equilibrium, exchanging energy between the two blocks one unit at a time — flip a random up-spin
and a random down-spin (with `numpy.random.default_rng`) — from a hot-$A$/cold-$B$ start, and
watch $N_\uparrow^A$ relax to the equilibrium value. The animation shows energy sloshing to the
peak.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    up_A[np.argmax(S_total)],
    up_tot * N_A / (N_A + N_B),
    "the total entropy is maximized at the proportional partition — the second law from counting",
    atol=1,
)
validate.close(
    upA_trace[-30:].mean(),
    float(up_peak),
    "the simulated energy exchange relaxes to the entropy-maximizing partition",
    atol=3.0 * width,
)
validate.check(
    np.std(upA_trace[-30:]) < 4.0 * width,
    "the late-time fluctuations stay within the narrow 1/√N band about the peak",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6 — Temperature emerges

We have used the word "hotter" loosely; now we earn it. At the equilibrium peak the total
entropy is flat, $dS_{\rm total}/dN_\uparrow^A=0$, which means $\partial S_A/\partial E_A=
\partial S_B/\partial E_B$ {eq}`eq-temperature`. Some quantity, the slope of each magnet's
entropy curve, has become equal — and energy stops flowing precisely when it does. *That* is
what temperature is. We **define** $1/T=\partial S/\partial E$, and everything we expect of
temperature follows: two bodies in contact reach a common $\partial S/\partial E$ (they reach a
common temperature), and before they do, the body with the *smaller* slope (the larger $T$, the
hotter one) gives up energy to the body with the larger slope, because that transfer raises the
total entropy ({numref}`fig-me-temperature`). Temperature was never an independent concept; it
is the energy-derivative of a count.

**This exercise (worked).** Compute $\partial S/\partial E$ for each block with `numpy.gradient`
(with $E=N_\uparrow\varepsilon$ and $\varepsilon=1$, this is $\partial S/\partial N_\uparrow$),
plot the two curves against the shared energy axis, and confirm they cross at the equilibrium
partition. For equal blocks that equality is guaranteed by symmetry, so let the *unequal* pair
of Exercise 4 ($N_A=50$, $N_B=150$, $N_\uparrow=80$) carry the real test: two genuinely
different entropy curves whose slopes nonetheless agree at the equilibrium partition — where,
by definition, the temperatures are equal.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    slope_A2,
    slope_B2,
    "two genuinely different entropy curves reach equal ∂S/∂E at equilibrium — this defines 1/T",
    atol=2e-2,
)
validate.close(
    slope_A,
    slope_B,
    "the equal blocks agree too (by symmetry): ∂S_A/∂E_A = ∂S_B/∂E_B at the peak",
    atol=3e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — The Boltzmann distribution from a reservoir

This is the formula the rest of statistical mechanics runs on, and we are going to *count* it
rather than assume it. Take a tiny system — with energy levels $0,1,2,\dots$ — in contact with a
large **reservoir**, itself a paramagnet holding many units of energy ({numref}`fig-me-reservoir`).
The combined system is isolated, so by the postulate every joint microstate is equally likely.
The number of joint microstates with the small system at energy $n$ is just the reservoir's
multiplicity with the rest of the energy, $\Omega_{\rm res}(Q-n)$, so $P(n)\propto\Omega_{\rm
res}(Q-n)$ {eq}`eq-boltzmann`. Now the magic: handing $n$ units to the small system lowers
$\ln\Omega_{\rm res}$ by very nearly a straight line, with slope $\partial S_{\rm res}/\partial
E=1/kT$, so $\Omega_{\rm res}(Q-n)\propto e^{-n/kT}$. The probability falls off exponentially in
energy — the **Boltzmann factor** — and we obtained it purely by counting the reservoir's states.

**This exercise (worked).** For a paramagnet reservoir of $N_R=5000$ spins holding $Q=1500$
units of energy (below half-filling, a positive temperature), compute $P(n)\propto\Omega_{\rm
res}(Q-n)$ by direct counting in log space, compare it to $e^{-\beta n}$ with $\beta=\partial
S_{\rm res}/\partial E=\ln[(N_R-Q)/Q]$, and confirm they agree to about a part in $10^3$.
*Frame:* we did not assume the Boltzmann distribution — we counted it.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    P_counted,
    P_boltzmann,
    "P(E) ∝ e^(−E/kT) falls out of counting the reservoir's microstates — the Boltzmann distribution",
    atol=2e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 8 — Negative temperature and the limits of the picture

Defining temperature as $1/T=\partial S/\partial E$ has a startling consequence worth meeting,
because it sharpens what temperature really is. So far we have worked *below* half-filling,
where the paramagnet behaves normally — adding energy raises the entropy, $\partial S/\partial
E>0$, an ordinary positive temperature. But the paramagnet's spectrum is **bounded**: there is a
maximum energy, all spins up. Its entropy therefore peaks at half-filling and then *falls*
({numref}`fig-me-negT`). Past the peak — a population-inverted state, more spins up than down —
adding energy *decreases* the entropy, so $\partial S/\partial E<0$, a formally **negative
temperature**. Far from being colder than absolute zero, such a state is *hotter than infinite*:
it gives up energy to anything with a positive temperature. A system with an *unbounded* energy
spectrum never does this; its entropy rises forever. The lesson is that $1/T=\partial S/\partial
E$, not "average energy," is the true definition of temperature.

**This exercise (student).** For a paramagnet of $N=200$ spins, compute $S/k=\ln\binom{N}{N_
\uparrow}$ with `scipy.special.gammaln` across the full energy range, take $\partial S/\partial
E$ with `numpy.gradient`, and confirm it is positive below half-filling and *negative* above it.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.check(
    below_half > 0 and above_half < 0,
    "a bounded spectrum admits negative temperature — 1/T = ∂S/∂E is the true definition",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 9 — Counting is statistical mechanics

Look back at what one postulate and the counting of three notebooks have built. We assumed only
that every accessible microstate of an isolated system is equally likely — the **microcanonical
ensemble**. From the binomial multiplicity of a genuinely classical system, the two-state
paramagnet ([§5.1](counting.ipynb)), computed in the log space the counts forced on us ([§5.3](large-n-limit.ipynb)), we read off
**entropy** as $S=k\ln\Omega$. Bringing two magnets into contact, the $1/\sqrt N$ sharpness of
[§5.3](large-n-limit.ipynb) made the equilibrium partition overwhelmingly probable, and the drift toward it *is* the
**second law**. The slope of entropy that equalizes there *is* **temperature**. And counting a
reservoir's leftover states gave the **Boltzmann factor** $e^{-E/kT}$, the formula everything
downstream is built from. Not one of these was assumed as physics; each was counted. We have
arrived at the foundations of thermodynamics — entropy, irreversibility, temperature — and found
them to be combinatorics all along.

**This exercise.** Close with the unifying check that the energy-derivative of entropy
(Exercise 6's definition of $1/T$) and the Boltzmann exponent (Exercise 7's $\beta$) are the
*same* quantity for the paramagnet: confirm with `np.allclose` that the entropy slope
$\partial S/\partial E$ of a paramagnet at $N_\uparrow=Q$ equals the reservoir exponent
$\ln[(N-Q)/Q]$ — the single $1/kT$ that ties counting to temperature.

In [ ]:
# (solution hidden on the public site)


### Validation 9

In [ ]:
validate.close(
    beta_slope,
    beta_boltz,
    "the entropy slope ∂S/∂E and the Boltzmann exponent are one quantity, 1/kT — counting becomes temperature",
    rtol=1e-3,
)

## Notebook summary

From one postulate — the microcanonical ensemble — and the counting of [§5.1](counting.ipynb)–[§5.3](large-n-limit.ipynb), this notebook
built the conceptual core of statistical mechanics, and computed every step exactly with a
genuinely classical system, the two-state paramagnet.

- **Microstates and the postulate** {eq}`eq-postulate`, {eq}`eq-micro-macro`: every accessible
  microstate of an isolated system is equally likely (the **microcanonical ensemble**), so a
  macrostate's probability is its multiplicity.
- **The paramagnet** {eq}`eq-paramagnet`: $\Omega(N,N_\uparrow)=\binom{N}{N_\uparrow}$, *exactly*
  the binomial / combinations count of [§5.1](counting.ipynb), computed as $\ln\Omega$ via `scipy.special.gammaln`
  because the count overflows (the [§5.3](large-n-limit.ipynb) lesson).
- **Entropy** {eq}`eq-microstates-entropy`: $S=k\ln\Omega$ is extensive ($\ln\Omega(2N,2N_\uparrow)\to2\ln
  \Omega(N,N_\uparrow)$) and maximal at half-filling ($S_{\max}=\ln\binom{100}{50}\approx66.8$).
- **The second law** {eq}`eq-second-law`: $\Omega_{\rm total}(N_\uparrow^A)=\Omega_A\Omega_B$ is
  sharply peaked (the $1/\sqrt N$ sharpness) at proportional energy sharing ($N_\uparrow^A=30$ for
  equal magnets, $20$ for $50/150$); energy relaxes there via spin flips, and that drift *is* the
  second law.
- **Temperature** {eq}`eq-temperature`: at the peak $\partial S_A/\partial E_A=\partial S_B/
  \partial E_B$; $1/T=\partial S/\partial E$ defines temperature.
- **The Boltzmann distribution** {eq}`eq-boltzmann`: counting a paramagnet reservoir's leftover
  states gives $P(E)\propto e^{-E/kT}$, with $\beta=\ln[(N_R-Q)/Q]$ — the master formula, counted
  not assumed. The bounded paramagnet spectrum even admits negative temperature past
  half-filling, underscoring that $1/T=\partial S/\partial E$ is the true definition.

We have arrived at the foundations of thermodynamics by counting. But why is an average over all
of phase space the right thing to compute? [§5.5](ergodicity.ipynb) next examines the **ergodic hypothesis** — that a
single trajectory, given time, samples the whole energy surface — which licenses the ensemble
method; the **canonical ensemble**, summing the Boltzmann factor into the partition function,
follows later in the volume.

## Outlook

- **Why the ensembles are valid: ergodicity ([§5.5](ergodicity.ipynb)).** What we measure is a time average along one
  trajectory; what an ensemble computes is an average over phase space. The ergodic hypothesis
  says they agree — demonstrable for a chaotic system, and breakable for an integrable one.
- **The canonical ensemble.** Summing the Boltzmann factor counted here over a system's states
  into the partition function $Z$ turns $\ln Z$ into a generating function for energy, entropy,
  free energy, and heat capacity — the machinery built later in the volume on this notebook's
  Boltzmann bridge.
- **The other derivatives of entropy ([§5.6](ideal-gas-fundamental-relation.ipynb)).** Pressure and chemical potential as $\partial S/\partial
  V$ and $\partial S/\partial N$, and the fundamental thermodynamic relation $dE=T\,dS-P\,dV+\mu
  \,dN$ — the rest of thermodynamics from the same $S=k\ln\Omega$.
- **Monte Carlo and the Ising model ([§5.10](ising-emergence-universality.ipynb)).** The spin-exchange simulation here is the seed of the
  Metropolis algorithm and the interacting Ising magnet, whose heat capacity genuinely diverges
  at a phase transition — the capstone of the volume.
- **Quantum statistics (Volume VII).** The same counting for *indistinguishable* particles gives
  Fermi–Dirac and Bose–Einstein — once Volume VI builds the quantum mechanics behind it.
- **Cross-reference** [§5.1](counting.ipynb) (the binomial count), [§5.3](large-n-limit.ipynb) (log space, entropy, the $1/\sqrt N$ sharpness).

In [ ]:
from ecp.style import footer

footer()